In [ ]:
# -*- coding: utf-8 -*-

Copy of Qwen3_VL_Two_Stage_Agentic_Invoice_Pipeline.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1peFofwZzhl-B8YbFTwbCWuNGqHu3eMEV

# Invoice Extraction — Qwen3-VL-4B-Instruct

Single-invoice testing notebook (no batch/evaluation layer) — upload one invoice,
run it, look at the result, repeat.

**What changed from the earlier Qwen2.5-VL-3B notebook:**
- **Model**: Qwen3-VL-4B-Instruct (newer generation, stronger text/spatial reasoning per Alibaba's release notes) instead of Qwen2.5-VL-3B.
- **Two free-form generation calls per invoice** (header fields, then the line-items table), each parsed and JSON-repaired independently — no constrained decoding, no `outlines` dependency. See the revision-2 changelog below for why.
- **`max_pixels` raised** (from ~800k to ~1600k) based on what the dense 9-column IndoFab invoice suggested — wide/dense tables need more resolution than simpler invoices.
- **Schema**: added `item_code` (separate from `hsn_code`) and `other_charges` (a generic charge line distinct from `shipping_charges`, so uniquely-named extra charges don't just vanish into `additional_fields` uncounted).
- **Prompt**: vendor/customer and subtotal rules are now *definitional* ("vendor = the issuing/selling party, regardless of label") rather than a list of label synonyms — since we found labels like "Ship To" mean different things on different templates.
- **Removed**: batch/folder processing, the evaluation report (self-consistency across samples, per-field accuracy, CSV export). You're testing one invoice at a time and pasting me the output directly.

## Bugfix changelog (this revision)

Applied on top of the previous version, no architecture changes:
- **Pinned `outlines==0.1.11`** so constrained decoding actually loads instead of silently falling back every run.
- **Fixed vendor/customer GSTIN collision**: `regex_fallback` now tracks match position so a customer-field lookup can't re-find the vendor's text, and a hard invariant check now flags `vendor_customer_gstin_identical` if it ever happens anyway.
- **Wired in `vendor_phone`/`vendor_email` validation** (patterns existed but were never actually called).
- **`needs_review` now reflects `unverifiable_tax_breakdown` line items**, missing line items, and missing core identity fields (vendor/customer name, invoice number, total) -- previously these could pass silently.
- **Fixed `subtotal`/`total_amount` reconciliation** to stop double-subtracting `discount_total` (subtotal is defined as post-discount per the extraction prompt) and stop mixing pre-tax/post-tax bases when summing line items.
- **Fixed the `"discount"` column alias** so a bare "Discount" header maps to the amount field instead of the percentage field.
- **`_clean_numeric` now handles `"Rs."`/currency-symbol prefixes and the Indian `"/-"` ("only") suffix correctly** -- these previously caused silent parse failures (returned `None` for a valid number) or, for `"/-"`, a wrongly-negated value.
- **`_clean_numeric` now handles accounting-style negatives**, e.g. `"(1,234.56)"` -> `-1234.56`.
- **`generate_invoice`/`extract_invoice` wrapped for robustness**: a bad tokenizer pad token or an unexpected generation error now returns a reviewable result instead of crashing the run.
- **`field_sources` only marks fields the model actually returned**, instead of defaulting every core field to `"llm"` regardless of whether it was null.

GSTIN checksum was audited and left unchanged -- verified correct against two independently-published worked examples.


## Revision 2 changelog

- **Removed `outlines`/constrained decoding entirely.** It was the main
  reason generation was slow (loading the vision+FSM integration, plus the
  FSM having to track open-ended types like `bank_details`/`additional_fields`
  even after the line-items array was split out). The pipeline now always
  makes two plain `transformers.generate()` calls per invoice — header, then
  line items — each parsed with the same JSON-repair logic the old fallback
  path already had. This *is* the "fallback path" from revision 1, just made
  the only path, with the header/line-items split kept because it helps
  accuracy on dense tables, not because it's faster to constrain.
- **`bank_details` is now a structured object**: `account_holder_name`,
  `account_number`, `ifsc_code`, `bank_name`, `branch`, `upi_id` — instead of
  a free-text-or-dict blob. `ifsc_code`/`account_number`/`upi_id` get their
  own regex fallback patterns, used ONLY to fill a sub-field the model left
  null — an already-extracted value is never overwritten.
- **Line items now carry `cgst_rate`/`sgst_rate`/`igst_rate`** (percentages)
  alongside the existing `cgst_amount`/`sgst_amount`/`igst_amount`, and
  `reconcile_line_item` uses the rates as an extra arithmetic-check candidate.
- **`due_date` is computed from `payment_terms` + `invoice_date`** ("Net 30",
  "45 days from invoice date", "Due on receipt", etc.) but ONLY when the
  invoice has no due date printed on it — a due date the model actually read
  off the page is never overridden by the computed one.
- **`customer_pan`** added to the schema, validated the same way as
  `vendor_pan`, and derived from `customer_gstin` (characters 3–12 of the
  GSTIN) whenever the model didn't return a valid PAN directly — same
  treatment `vendor_pan` already got in revision 1.

## 1. Install dependencies

In [2]:
!pip install -q -U "transformers>=4.51" accelerate bitsandbytes pydantic python-dateutil "qwen-vl-utils[decord]" "pillow>=10.1" pymupdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 119.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 117.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 78.9 MB/s eta 0:00:00


## 2. Imports

In [3]:
import re
import json
import copy
import torch
from typing import Optional, List, Dict, Any, Union
from PIL import Image
from pydantic import BaseModel, Field, ValidationError
from dateutil import parser as dateparser
from transformers import (
    AutoProcessor,
    AutoTokenizer,
    AutoModelForCausalLM,
    Qwen3VLForConditionalGeneration,
    BitsAndBytesConfig,
)
from qwen_vl_utils import process_vision_info

torch.manual_seed(0)

## 3. Load Qwen3-VL-4B-Instruct (4-bit)

Plain `transformers` load — no `outlines`, no constrained decoding. Section 6
makes two separate free-form generation calls per invoice (header fields,
then the line-items table), each parsed and JSON-repaired independently.

In [4]:
MODEL_NAME_VL = "Qwen/Qwen3-VL-4B-Instruct"
MODEL_NAME_TEXT = "Qwen/Qwen3-4B-Instruct-2507"

# One shared 4-bit config is intentionally used for BOTH models.
# No model is deleted/offloaded between stages. Both stay resident.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 1600 * 28 * 28

# -----------------------------
# Stage 1 model: vision-language
# -----------------------------
processor = AutoProcessor.from_pretrained(
    MODEL_NAME_VL,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)
hf_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_NAME_VL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
hf_model.eval()

# -----------------------------
# Stage 2 model: text reasoning
# -----------------------------
text_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_TEXT)
text_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_TEXT,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
text_model.eval()

print("Both models are loaded simultaneously in 4-bit.")
print("VL model :", MODEL_NAME_VL)
print("Text model:", MODEL_NAME_TEXT)

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Both models are loaded simultaneously in 4-bit.
VL model : Qwen/Qwen3-VL-4B-Instruct
Text model: Qwen/Qwen3-4B-Instruct-2507


## 4. Schema

- **`other_charges`** — a generic additional-charge line distinct from
  `shipping_charges`, so a uniquely-named charge doesn't just disappear into
  `additional_fields` uncounted in reconciliation.
- **`LineItem`** — description, hsn_code, quantity, unit_price, discount,
  taxable_amount, `cgst_rate`/`cgst_amount`, `sgst_rate`/`sgst_amount`,
  `igst_rate`/`igst_amount`, total. Rate + amount are both kept per tax type
  (rather than one combined `tax_rate_pct`) so a mixed CGST+SGST/IGST invoice
  never has to guess which rate belongs to which amount.
- **`bank_details`** — structured (`account_holder_name`, `account_number`,
  `ifsc_code`, `bank_name`, `branch`, `upi_id`) instead of a free-text blob,
  so downstream systems can consume it field-by-field.
- **`customer_pan`** — same treatment as `vendor_pan`: validated by format,
  and derivable from the corresponding GSTIN if the model doesn't return it
  directly.
- **Header/line-items split** — `InvoiceHeaderSchema` (everything except
  `line_items`) gets its own focused prompt+call; `line_items` gets a second,
  separate prompt+call. Both are free-form generation (no constrained
  decoding) -- the split is kept purely because a focused prompt does better
  on dense item tables, not for FSM performance.

In [5]:
class BankDetails(BaseModel):
    account_holder_name: Optional[str] = None
    account_number: Optional[str] = None
    ifsc_code: Optional[str] = None
    bank_name: Optional[str] = None
    branch: Optional[str] = None
    upi_id: Optional[str] = None


class LineItem(BaseModel):
    description: Optional[str] = None
    hsn_code: Optional[str] = None          # official HSN/SAC tax code, always numeric
    quantity: Optional[float] = None
    unit_price: Optional[float] = None
    discount: Optional[float] = None
    taxable_amount: Optional[float] = None  # pre-tax line amount
    cgst_rate: Optional[float] = None       # percentage, e.g. 9 for 9%
    cgst_amount: Optional[float] = None
    sgst_rate: Optional[float] = None
    sgst_amount: Optional[float] = None
    igst_rate: Optional[float] = None
    igst_amount: Optional[float] = None
    total: Optional[float] = None           # post-tax line amount


class InvoiceSchema(BaseModel):
    invoice_number: Optional[str] = None
    invoice_date: Optional[str] = None
    due_date: Optional[str] = None
    po_number: Optional[str] = None

    vendor_name: Optional[str] = None
    vendor_address: Optional[str] = None
    vendor_gstin: Optional[str] = None
    vendor_pan: Optional[str] = None
    vendor_cin: Optional[str] = None
    vendor_phone: Optional[str] = None
    vendor_email: Optional[str] = None

    customer_name: Optional[str] = None
    customer_address: Optional[str] = None
    customer_gstin: Optional[str] = None
    customer_pan: Optional[str] = None

    payment_terms: Optional[str] = None
    bank_details: Optional[BankDetails] = None

    line_items: List[LineItem] = Field(default_factory=list)

    subtotal: Optional[float] = None
    discount_total: Optional[float] = None
    tax_total: Optional[float] = None
    shipping_charges: Optional[float] = None
    other_charges: Optional[float] = None
    round_off: Optional[float] = None
    total_amount: Optional[float] = None
    currency: Optional[str] = None

    additional_fields: Dict[str, Any] = Field(default_factory=dict)


# Header/line-items split kept from revision 1 -- not for constrained-decoding
# performance anymore (outlines is gone), but because a focused prompt+schema
# per call (header fields vs. the items table) still helps accuracy on dense
# tables. Derived with create_model so it can't drift out of sync with
# InvoiceSchema.
from pydantic import create_model

_header_field_defs = {
    name: (field.annotation, field)
    for name, field in InvoiceSchema.model_fields.items()
    if name != "line_items"
}
InvoiceHeaderSchema = create_model("InvoiceHeaderSchema", **_header_field_defs)

_header_schema_example = InvoiceHeaderSchema().model_dump()
HEADER_SCHEMA_JSON_EXAMPLE = json.dumps(_header_schema_example, indent=2)
LINE_ITEM_SCHEMA_JSON_EXAMPLE = json.dumps(LineItem().model_dump(), indent=2)

CORE_FIELDS = [f for f in InvoiceSchema.model_fields.keys() if f not in ("line_items", "additional_fields")]

# Schema example shown to the model includes ONE fully-populated line item (not an
# empty list) -- otherwise the model never sees the real field names and invents
# its own column names instead.
_schema_example = InvoiceSchema().model_dump()
_schema_example["line_items"] = [LineItem().model_dump()]
SCHEMA_JSON_EXAMPLE = json.dumps(_schema_example, indent=2)
print(SCHEMA_JSON_EXAMPLE)

{
  "invoice_number": null,
  "invoice_date": null,
  "due_date": null,
  "po_number": null,
  "vendor_name": null,
  "vendor_address": null,
  "vendor_gstin": null,
  "vendor_pan": null,
  "vendor_cin": null,
  "vendor_phone": null,
  "vendor_email": null,
  "customer_name": null,
  "customer_address": null,
  "customer_gstin": null,
  "customer_pan": null,
  "payment_terms": null,
  "bank_details": null,
  "line_items": [
    {
      "description": null,
      "hsn_code": null,
      "quantity": null,
      "unit_price": null,
      "discount": null,
      "taxable_amount": null,
      "cgst_rate": null,
      "cgst_amount": null,
      "sgst_rate": null,
      "sgst_amount": null,
      "igst_rate": null,
      "igst_amount": null,
      "total": null
    }
  ],
  "subtotal": null,
  "discount_total": null,
  "tax_total": null,
  "shipping_charges": null,
  "other_charges": null,
  "round_off": null,
  "total_amount": null,
  "currency": null,
  "additional_fields": {}
}


## 5. Prompt

Vendor/customer and subtotal rules are definitional rather than a label list —
"Ship To" means different things on different templates, so a definitional
rule gives the model a chance to reason about role rather than pattern-match
a label. Also covers: structured `bank_details` sub-fields, per-tax-type
`cgst_rate`/`sgst_rate`/`igst_rate`, and copying `payment_terms` verbatim
(needed downstream to compute `due_date` when it isn't printed).

In [6]:
SYSTEM_PROMPT_HEADER = """You are a precise invoice-data-extraction engine reading an invoice IMAGE directly.

Rules:
1. Invoices vary widely in layout, field labels, language, and image quality. Do NOT assume any fixed position or label wording -- read the image semantically, using table structure and spatial layout to disambiguate which number belongs to which field.

2. "vendor" means the ISSUING/SELLING party -- the entity that generated and is sending this invoice, normally in the letterhead at the top. "customer" means whichever party is being BILLED / charged, i.e. whoever owes the money on this invoice. Do NOT rely on the literal label to decide this -- labels like "Bill To", "Sold By", "Party Name", or even "Ship To" are used inconsistently across templates. Reason about which party is issuing the invoice and which party is being charged, not which label is attached. vendor_gstin and customer_gstin should normally be DIFFERENT values.

3. The pre-tax total of all line items may be labeled "Subtotal", "Taxable Amount", "Taxable Value", or "Assessable Value" depending on the template -- these are the same concept. Map whichever one appears to the "subtotal" field.

4. If the invoice has an additional charge line that is NOT shipping/freight and NOT a line item (e.g. a service charge, handling fee, or similar), put its amount in "other_charges".

5. "bank_details" is a structured object, not free text: account_holder_name, account_number, ifsc_code, bank_name, branch, upi_id. Fill each sub-field independently from wherever it appears on the invoice (often a "Bank Details"/"Payment Details" box). Copy account_number, ifsc_code, and upi_id EXACTLY as printed. Leave a sub-field null if it isn't visible -- do not guess one from another.

6. "payment_terms" should be copied close to verbatim (e.g. "Net 30", "45 days from invoice date", "Due on receipt", "50% advance") -- do not paraphrase it, since the exact wording is used later to compute a due date when none is printed.

7. "due_date" is only for a due date actually PRINTED on the invoice. If none is printed, leave it null -- do not calculate one yourself.

8. customer_pan should be extracted the same way as vendor_pan if a customer PAN is visible on the invoice (independent of GSTIN).

9. Output ALL numbers as plain numbers with no currency symbols, no thousands separators (commas), and no units -- e.g. write 30134.00, not "Rs. 30,134.00".

10. If a schema field is not visible in the image, set it to null. NEVER invent, guess, or hallucinate a value.

11. If you see information that doesn't map to any schema field, put it under "additional_fields" as {"your_label": "value"} instead of discarding it.
11A. Extract EVERY meaningful piece of information visible in the invoice. If information does not map to a canonical field, preserve it under "additional_fields" using the original printed label whenever possible. NEVER discard information merely because the schema has no dedicated field.

11B. TAX EXTRACTION IS HIGH-PRIORITY. Identify and preserve every tax/accounting value and label visible anywhere on the page, including CGST, SGST, IGST, CESS, Input CGST, Input SGST, Input IGST, Input CESS, TDS, TCS, Tax Payable, GST Payable, Net Tax Payable, Tax Liability, tax rates, tax bases, tax amounts, deductions, credits, reversals, exemptions and other tax notes. "CGST" and "Input CGST" are different concepts. "TDS" is not GST.

11C. Put non-canonical tax/accounting information into "additional_fields.tax_details" while also preserving the original printed label/value. A tax detail should contain enough context to preserve the source meaning, for example:
{"label_as_printed":"Input CGST","semantic_type":"input_cgst","rate":null,"base_amount":null,"amount":1234.50,"source":"extracted"}
Do not invent a rate/base/amount that is not visible. Use null when not present.

11D. Preserve every other non-canonical field in additional_fields without overwriting unrelated keys.

12. Copy identifiers (GSTIN, PAN, CIN, phone, invoice number) EXACTLY as printed, character by character. Only dates get normalized, to YYYY-MM-DD.

13. This schema does NOT include line items -- they are extracted separately. Do not include a "line_items" key.

14. Output ONLY a single valid JSON object matching the schema below. No markdown fences, no commentary before or after.

Schema:
""" + HEADER_SCHEMA_JSON_EXAMPLE

SYSTEM_PROMPT_LINE_ITEMS = """You are a precise invoice-line-item-extraction engine reading an invoice IMAGE directly.

Rules:
1. Use the EXACT field names shown in the schema below. If a column shows an official HSN/SAC tax classification code (e.g. "997159", always numeric), put it in "hsn_code"; ignore any separate internal SKU/part code column if present -- it is not captured.

2. Extract EVERY line item row visible in the items table, top to bottom. Fill cgst_amount/sgst_amount/igst_amount for EVERY row that has a GST%/CGST/SGST/IGST breakdown -- never leave these null just because "total" is filled. Also fill the matching cgst_rate/sgst_rate/igst_rate as a plain percentage number (e.g. 9 for "9%" or "CGST@9%") wherever that rate is shown or printed in a column header shared by the whole table. "taxable_amount" is the line amount BEFORE tax; "total" is the final line amount AFTER tax.

3. Output ALL numbers as plain numbers with no currency symbols, no thousands separators (commas), and no units.

4. If a schema field is not visible for a row, set it to null. NEVER invent, guess, or hallucinate a value.

5. Copy identifiers (HSN codes) EXACTLY as printed, character by character.
6. Preserve any row-level tax/accounting information that does not fit the canonical line-item schema in that row's "additional_fields", including printed labels such as Input CGST, TDS, TCS, CESS, tax payable and custom charges.

6. Output ONLY a single valid JSON ARRAY of line-item objects -- e.g. [{...}, {...}] -- one element per row, in the order they appear on the invoice. No markdown fences, no commentary before or after, no wrapping object.

Schema for each array element:
""" + LINE_ITEM_SCHEMA_JSON_EXAMPLE


def build_messages_header(image_path: str):
    return [
        {"role": "system", "content": SYSTEM_PROMPT_HEADER},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path, "min_pixels": MIN_PIXELS, "max_pixels": MAX_PIXELS},
                {"type": "text", "text": "Extract the invoice header data from this image (everything except line items). Return only the JSON object."},
            ],
        },
    ]


def build_messages_line_items(image_path: str):
    return [
        {"role": "system", "content": SYSTEM_PROMPT_LINE_ITEMS},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path, "min_pixels": MIN_PIXELS, "max_pixels": MAX_PIXELS},
                {"type": "text", "text": "Extract every line item row from this invoice's items table. Return only the JSON array."},
            ],
        },
    ]

## 6. Generation — two free-form calls, JSON-repaired

No `outlines`, no constrained decoding. Each invoice gets two plain
`model.generate()` calls: one for the header fields (`SYSTEM_PROMPT_HEADER`),
one for the line-items table (`SYSTEM_PROMPT_LINE_ITEMS`). Both are parsed
with the same fence-stripping / brace-matching / truncated-JSON-repair logic
from section 7 -- this was already the fallback path in revision 1, it's now
just the only path.

In [7]:
def _extract_json_array_block(text):
    start = text.find("[")
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "[":
            depth += 1
        elif text[i] == "]":
            depth -= 1
            if depth == 0:
                return text[start:i + 1]
    return None


def _parse_line_items_freeform(raw_text):
    """Parses+repairs the free-form line-items array: strip fences, extract
    the bracketed block, parse, then normalize/validate each row -- dropping
    only the individual rows that don't validate rather than the whole list."""
    cleaned = strip_code_fences(raw_text)
    candidate = _extract_json_array_block(cleaned) or cleaned
    try:
        parsed = json.loads(candidate)
    except json.JSONDecodeError:
        return []
    if not isinstance(parsed, list):
        return []
    items = []
    for li in parsed:
        norm = normalize_line_item(li)
        try:
            items.append(LineItem(**norm).model_dump())
        except ValidationError:
            continue
    return items


def _run_generation(image_path: str, messages_fn, max_new_tokens: int) -> str:
    """One free-form generation call: builds the prompt from the given
    message-builder, runs the model, and returns the raw decoded text."""
    messages = messages_fn(image_path)
    text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_prompt], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(hf_model.device)

    # Some tokenizers don't define a pad token; fall back to eos so .generate()
    # doesn't raise or silently mis-pad.
    pad_id = processor.tokenizer.pad_token_id
    if pad_id is None:
        pad_id = processor.tokenizer.eos_token_id

    with torch.no_grad():
        out = hf_model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=pad_id,
        )
    input_len = inputs["input_ids"].shape[1]
    return processor.batch_decode(out[:, input_len:], skip_special_tokens=True)[0]


def generate_invoice(image_path: str, max_new_tokens: int = 2048):
    """Two free-form generation calls -- header fields, then the line-items
    table -- each parsed and JSON-repaired independently.

    Returns (data_dict_or_None, raw_texts_dict). data is None only if the
    header call's output couldn't be parsed/repaired into anything usable --
    the caller falls back to rescue_via_reread in that case. raw_texts is
    always returned so a failure is still inspectable.
    """
    header_raw = _run_generation(image_path, build_messages_header, max_new_tokens)
    header_parsed = safe_json_parse(header_raw)
    header_data = loosely_validate_header(header_parsed) if header_parsed else None

    li_raw = _run_generation(image_path, build_messages_line_items, max_new_tokens)
    line_items = _parse_line_items_freeform(li_raw)

    raw_texts = {"header_raw": header_raw, "line_items_raw": li_raw}

    if header_data is None:
        return None, raw_texts

    header_data["line_items"] = line_items
    # Add structured lossless tax metadata without deleting any VLM fields.
    header_data = ensure_lossless_tax_metadata(header_data)
    return header_data, raw_texts

## 7A. Stage 2 — Qwen3-4B full-invoice accounting + GST reconciliation

Stage 2 receives the **entire Stage-1 VLM extracted JSON (`o1`) in one payload**. It does not receive only vendor_name, descriptions, or one line at a time. The complete JSON is serialized into the LLM user message so Qwen3-4B can use header fields, all line items, HSN/SAC, quantities, amounts, discounts, taxable values, CGST/SGST/IGST, invoice totals, bank/payment fields, and any other extracted field.

Qwen3-4B returns one `line_items` result array in the same order as the Stage-1 JSON. Stage-1 monetary and tax extraction remains authoritative; Stage 2 adds account classification, Zoho tax mapping, confidence, and review metadata.


### Updated Stage-2 architecture: COMPLETE VLM JSON → ONE LLM CALL

The previous implementation split the invoice into individual line-item prompts. This notebook now sends the **complete Qwen3-VL JSON object** to Qwen3-4B in a single Stage-2 generation call. This is intentional: invoice-level context can affect classification and tax reconciliation, and no extracted fields are hidden from the LLM.

The live Zoho Chart of Accounts and live Zoho tax records are sent alongside the complete Stage-1 JSON as reference data.


In [ ]:
# ============================================================
# STAGE 2 — LIVE ZOHO COA + COMPLETE GST/TAX RECONCILIATION
# ============================================================

import json
import re
import copy
import torch
from typing import Optional, List, Dict, Any


def _as_float(value):
    """Safely convert invoice numeric values to float."""
    if value is None or value == "":
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None



def _merge_dict_preserve(existing, updates):
    """Deep-ish merge that never discards existing non-null values."""
    result = copy.deepcopy(existing) if isinstance(existing, dict) else {}
    if not isinstance(updates, dict):
        return result
    for key, value in updates.items():
        if key not in result:
            result[key] = copy.deepcopy(value)
        elif isinstance(result[key], dict) and isinstance(value, dict):
            result[key] = _merge_dict_preserve(result[key], value)
        elif result[key] in (None, "", [], {}):
            result[key] = copy.deepcopy(value)
    return result


def ensure_lossless_tax_metadata(invoice: dict) -> dict:
    """
    Add a structured tax_details layer while preserving the raw VLM
    additional_fields exactly. This is additive and lossless.
    """
    if not isinstance(invoice, dict):
        return invoice

    extras = invoice.get("additional_fields")
    if not isinstance(extras, dict):
        extras = {}
        invoice["additional_fields"] = extras

    tax_details = extras.get("tax_details")
    if not isinstance(tax_details, dict):
        tax_details = {}
        extras["tax_details"] = tax_details

    # Preserve any tax_details generated by the VLM. Also surface common
    # top-level/extra labels into structured semantic buckets without
    # deleting the originals.
    semantic_map = {
        "cgst": ("output_tax", "cgst"),
        "sgst": ("output_tax", "sgst"),
        "igst": ("output_tax", "igst"),
        "cess": ("output_tax", "cess"),
        "input cgst": ("input_tax_credit", "input_cgst"),
        "input sgst": ("input_tax_credit", "input_sgst"),
        "input igst": ("input_tax_credit", "input_igst"),
        "input cess": ("input_tax_credit", "input_cess"),
        "tds": ("tds", "amount"),
        "tcs": ("tcs", "amount"),
        "tax payable": ("tax_payable", "amount"),
        "gst payable": ("tax_payable", "amount"),
        "net tax payable": ("tax_payable", "amount"),
        "tax liability": ("tax_payable", "amount"),
    }

    def norm_label(label):
        return re.sub(r"[^a-z0-9]+", " ", str(label).lower()).strip()

    # Canonical line-level output tax information.
    output_tax = tax_details.setdefault("output_tax", {})
    for key, rate_key, amount_key in (
        ("cgst", "cgst_rate", "cgst_amount"),
        ("sgst", "sgst_rate", "sgst_amount"),
        ("igst", "igst_rate", "igst_amount"),
    ):
        rate = invoice.get(rate_key)
        amount = invoice.get(amount_key)
        if rate is not None or amount is not None:
            node = output_tax.setdefault(key, {})
            if rate is not None:
                node["rate"] = rate
            if amount is not None:
                node["amount"] = amount

    # Existing raw keys in additional_fields.
    for raw_key, raw_value in list(extras.items()):
        if raw_key == "tax_details":
            continue
        k = norm_label(raw_key)
        if k not in semantic_map:
            continue
        bucket, leaf = semantic_map[k]
        bucket_obj = tax_details.setdefault(bucket, {})
        if bucket == "tds":
            bucket_obj.setdefault("amount", raw_value)
        elif bucket == "tcs":
            bucket_obj.setdefault("amount", raw_value)
        elif bucket == "tax_payable":
            bucket_obj.setdefault("amount", raw_value)
        else:
            node = bucket_obj.setdefault(leaf, {})
            node.setdefault("amount", raw_value)

    # Always make these buckets available so downstream LLM logic sees a
    # stable semantic structure. Nulls mean "not observed", not zero.
    tax_details.setdefault("output_tax", {})
    tax_details.setdefault("input_tax_credit", {})
    tax_details.setdefault("tds", {})
    tax_details.setdefault("tcs", {})
    tax_details.setdefault("tax_payable", {})
    tax_details.setdefault("other_tax_information", [])

    return invoice


def _tax_snapshot(item: dict) -> dict:
    """Build a complete, authoritative tax snapshot from Stage-1 VLM output."""
    item = item if isinstance(item, dict) else {}
    extras = item.get("additional_fields") if isinstance(item.get("additional_fields"), dict) else {}
    tax_details = extras.get("tax_details") if isinstance(extras.get("tax_details"), dict) else {}

    cgst_rate = _as_float(item.get("cgst_rate"))
    cgst_amount = _as_float(item.get("cgst_amount"))
    sgst_rate = _as_float(item.get("sgst_rate"))
    sgst_amount = _as_float(item.get("sgst_amount"))
    igst_rate = _as_float(item.get("igst_rate"))
    igst_amount = _as_float(item.get("igst_amount"))

    tax_types = []
    if cgst_rate is not None or cgst_amount is not None:
        tax_types.append("CGST")
    if sgst_rate is not None or sgst_amount is not None:
        tax_types.append("SGST")
    if igst_rate is not None or igst_amount is not None:
        tax_types.append("IGST")

    # Preserve additional semantic tax concepts extracted by VLM.
    if tax_details.get("input_tax_credit"):
        tax_types.append("INPUT_TAX_CREDIT")
    if tax_details.get("tds"):
        tax_types.append("TDS")
    if tax_details.get("tcs"):
        tax_types.append("TCS")
    if tax_details.get("tax_payable"):
        tax_types.append("TAX_PAYABLE")
    if tax_details.get("output_tax", {}).get("cess") is not None:
        tax_types.append("CESS")

    # De-duplicate while preserving order.
    tax_types = list(dict.fromkeys(tax_types))

    calculated_tax = sum(
        value or 0.0
        for value in (cgst_amount, sgst_amount, igst_amount)
    )

    taxable_amount = _as_float(item.get("taxable_amount"))
    expected_tax = None
    tax_math_review = False

    gst_types_present = any(t in tax_types for t in ("CGST", "SGST", "IGST"))
    if taxable_amount is not None:
        expected_tax = 0.0
        for rate in (cgst_rate, sgst_rate, igst_rate):
            if rate is not None:
                expected_tax += taxable_amount * rate / 100.0
        if gst_types_present and abs(expected_tax - calculated_tax) > 2.0:
            tax_math_review = True

    total = _as_float(item.get("total"))
    line_total_review = False
    if total is not None and taxable_amount is not None and gst_types_present:
        if abs((taxable_amount + calculated_tax) - total) > 2.0:
            line_total_review = True

    return {
        "tax_present": bool(tax_types),
        "tax_types": tax_types,
        "cgst_rate": cgst_rate,
        "cgst_amount": cgst_amount,
        "sgst_rate": sgst_rate,
        "sgst_amount": sgst_amount,
        "igst_rate": igst_rate,
        "igst_amount": igst_amount,
        "input_tax_credit": copy.deepcopy(tax_details.get("input_tax_credit") or {}),
        "tds": copy.deepcopy(tax_details.get("tds") or {}),
        "tcs": copy.deepcopy(tax_details.get("tcs") or {}),
        "tax_payable": copy.deepcopy(tax_details.get("tax_payable") or {}),
        "other_tax_information": copy.deepcopy(tax_details.get("other_tax_information") or []),
        "calculated_tax_amount": round(calculated_tax, 2),
        "expected_tax_from_rates": None if expected_tax is None else round(expected_tax, 2),
        "tax_math_review": tax_math_review,
        "line_total_review": line_total_review,
        "zoho_tax_id": None,
        "zoho_tax_name": None,
        "zoho_tax_rate": None,
        "tax_confidence": 0.0 if (tax_math_review or line_total_review) else (0.95 if gst_types_present else 1.0),
        "tax_needs_review": bool(tax_math_review or line_total_review),
    }


def _invoice_tax_context(o1: dict) -> dict:
    """Build invoice-level financial/tax context for the LLM."""
    line_items = o1.get("line_items") or []

    aggregate = {
        "cgst_amount": 0.0,
        "sgst_amount": 0.0,
        "igst_amount": 0.0,
    }
    observed_rates = {"CGST": [], "SGST": [], "IGST": []}

    for item in line_items:
        if not isinstance(item, dict):
            continue
        for key in aggregate:
            aggregate[key] += _as_float(item.get(key)) or 0.0
        for tax_name, rate_key in (
            ("CGST", "cgst_rate"),
            ("SGST", "sgst_rate"),
            ("IGST", "igst_rate"),
        ):
            rate = _as_float(item.get(rate_key))
            if rate is not None and rate not in observed_rates[tax_name]:
                observed_rates[tax_name].append(rate)

    line_tax_total = round(sum(aggregate.values()), 2)

    return {
        "subtotal": o1.get("subtotal"),
        "discount_total": o1.get("discount_total"),
        "tax_total": o1.get("tax_total"),
        "shipping_charges": o1.get("shipping_charges"),
        "other_charges": o1.get("other_charges"),
        "round_off": o1.get("round_off"),
        "total_amount": o1.get("total_amount"),
        "currency": o1.get("currency"),
        "line_tax_totals": {
            "cgst_amount": round(aggregate["cgst_amount"], 2),
            "sgst_amount": round(aggregate["sgst_amount"], 2),
            "igst_amount": round(aggregate["igst_amount"], 2),
            "combined_line_tax": line_tax_total,
        },
        "observed_tax_rates": observed_rates,
        "additional_fields": o1.get("additional_fields") or {},
    }


def _tax_only_result(item: dict) -> dict:
    """Return a safe result when live Zoho COA was not supplied."""
    return {
        "ai_account_id": None,
        "ai_account_name": None,
        "ai_confidence": 0.0,
        "ai_needs_review": True,
        "final_account_id": None,
        "final_account_name": None,
        "tax_analysis": _tax_snapshot(item),
    }


def categorize_line_items(
    line_items: list,
    chart_of_accounts: list,
    vendor_name: str = None,
    invoice_context: dict = None,
    available_taxes: list = None,
    full_invoice_json: dict = None,
) -> list:
    """
    Stage 2 — send the COMPLETE Stage-1 VLM JSON to Qwen3-4B in ONE call.

    The previous implementation sent one partial line-item context at a time.
    This version deliberately does NOT do that. The full extracted invoice JSON
    is serialized unchanged and placed in the user message so the LLM can see:
      - vendor/customer/header fields
      - every line item
      - HSN/SAC
      - quantity/unit price/discount
      - taxable amounts
      - CGST/SGST/IGST rates AND amounts
      - subtotal/tax/total
      - bank/payment/additional fields
      - any other field Qwen3-VL extracted

    The LLM returns ONE JSON object containing a result for every line item.
    Stage-1 extraction values remain authoritative; Stage-2 may only add
    classification/reconciliation metadata around them.
    """
    line_items = line_items or []
    chart_of_accounts = chart_of_accounts or []
    available_taxes = available_taxes or []
    invoice_context = invoice_context or {}

    # Build the exact full invoice JSON that will be shown to Stage 2.
    # In the normal pipeline this is the actual Stage-1 `o1` object.
    if isinstance(full_invoice_json, dict):
        stage1_invoice = copy.deepcopy(full_invoice_json)
    else:
        # Compatibility path for the FastAPI endpoint that sends line_items
        # separately. This still gives the LLM the complete payload available
        # to that endpoint, but the local pipeline passes the real o1 object.
        stage1_invoice = copy.deepcopy(invoice_context)
        stage1_invoice["vendor_name"] = vendor_name
        stage1_invoice["line_items"] = copy.deepcopy(line_items)

    if not isinstance(stage1_invoice.get("line_items"), list):
        stage1_invoice["line_items"] = copy.deepcopy(line_items)

    # Live Zoho chart of accounts.
    accounts_map = {}
    coa_lines = []
    for idx, acc in enumerate(chart_of_accounts, 1):
        acc_id = str(acc.get("account_id") or acc.get("id") or f"ACC_{idx}").strip()
        acc_name = str(acc.get("account_name") or acc.get("name") or "Uncategorized").strip()
        acc_type = str(acc.get("account_type") or acc.get("type") or "expense").strip()
        accounts_map[acc_id] = acc_name
        accounts_map[acc_name.lower()] = (acc_id, acc_name)
        coa_lines.append({
            "account_id": acc_id,
            "account_name": acc_name,
            "account_type": acc_type,
        })

    # Live Zoho tax records.
    zoho_taxes = []
    for idx, tax in enumerate(available_taxes, 1):
        zoho_taxes.append({
            "tax_id": str(tax.get("tax_id") or tax.get("id") or f"TAX_{idx}").strip(),
            "tax_name": str(tax.get("tax_name") or tax.get("name") or "").strip(),
            "tax_rate": tax.get("tax_rate", tax.get("rate")),
            "tax_type": str(tax.get("tax_type") or tax.get("type") or "GST").strip(),
        })

    system_prompt = """
You are the Stage-2 accounting and Indian GST reconciliation engine.

IMPORTANT ARCHITECTURE RULE:
Stage 1 (Qwen3-VL) has already extracted the invoice. You are receiving the
ENTIRE extracted invoice JSON, not a summary and not selected fields.
Read the whole JSON before making any decision.

Your job has TWO independent responsibilities:
A) classify every invoice line against the supplied live Zoho Chart of Accounts;
B) analyze every tax component while preserving the VLM extraction values.

ACCOUNTING RULES
1. Select exactly one account for every non-empty invoice line.
2. The account_id and account_name MUST come from the supplied Zoho COA.
3. Never invent, rename, or modify a Zoho account.
4. Use the full invoice context plus line description, HSN/SAC, vendor, quantities,
   amounts and other fields as supporting evidence.
5. GST/TDS/tax is NOT the underlying expense category.
6. Never move a tax amount into "Other Expenses" merely because it is an extra
   amount on the bill. The chosen account must represent the underlying item or
   service being purchased.

TAX RULES
7. Inspect ALL tax fields present anywhere in the supplied JSON.
8. Preserve the Stage-1 VLM values exactly. Do not silently replace CGST, SGST,
   IGST rates or amounts with your own guesses.
9. Identify every tax type that is actually present.
10. Calculate combined line tax from the extracted tax amounts when possible.
11. Reconcile taxable amount, rates, tax amounts and line total when possible.
12. Reconcile invoice-level tax totals against the line-item tax totals when the
    data supports that check.
13. If a live Zoho tax list is supplied, map to the best matching record ONLY
    from that list. If no exact match is defensible, return null and set review.
14. Never collapse CGST+SGST into one generic tax field; retain individual
    components in the returned tax_analysis.
15. If tax information is absent, do not invent it.
16. If arithmetic is inconsistent or the tax breakdown is incomplete, set
    tax_needs_review=true.

OUTPUT RULES
17. Return ONLY one valid JSON object. No markdown and no explanation.
18. The top-level object MUST contain exactly:
    {
      "line_items": [ ... ]
    }
19. Return one result object for EVERY line item, in the SAME ORDER as the
    Stage-1 line_items array. Never drop a line.
20. Each result object must contain:
    - line_index (1-based)
    - source_description
    - account_id
    - account_name
    - confidence_score
    - ai_needs_review
    - tax_analysis
21. account_id/account_name are classification outputs only. The original
    extracted monetary/tax values remain in Stage 1 and are not to be rewritten.
22. tax_analysis must include the complete tax picture available for that line:
    tax_present, tax_types, cgst_rate, cgst_amount, sgst_rate, sgst_amount,
    igst_rate, igst_amount, taxable_amount, calculated_tax_amount,
    zoho_tax_id, zoho_tax_name, zoho_tax_rate, tax_confidence, tax_needs_review.
"""

    user_payload = {
        "stage1_extracted_invoice_json": stage1_invoice,
        "live_zoho_chart_of_accounts": coa_lines,
        "live_zoho_tax_records": zoho_taxes,
    }
    messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": (
                "Analyze this COMPLETE Stage-1 VLM invoice JSON in one pass. "
                "Do not ask for missing pieces and do not reduce the invoice to "
                "only the line-item descriptions.\n\n" +
                json.dumps(user_payload, ensure_ascii=False, indent=2)
            ),
        },
    ]
    try:
        text_prompt = text_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        inputs = text_tokenizer([text_prompt], return_tensors="pt").to(text_model.device)
        pad_id = text_tokenizer.pad_token_id
        if pad_id is None:
            pad_id = text_tokenizer.eos_token_id
        # One Stage-2 call for the whole invoice.
        with torch.inference_mode():
            outputs = text_model.generate(
                **inputs,
                max_new_tokens=max(1024, 320 * max(1, len(line_items))),
                do_sample=False,
                pad_token_id=pad_id,
            )
        generated = outputs[0][inputs.input_ids.shape[1]:]
        raw_output = text_tokenizer.decode(
            generated,
            skip_special_tokens=True,
        ).strip()
        print("\n" + "=" * 80)
        print("STAGE 2 — COMPLETE STAGE-1 JSON SENT TO QWEN3-4B")
        print("Number of Stage-1 line items:", len(line_items))
        print("Full Stage-1 JSON sent to LLM:")
        print(json.dumps(stage1_invoice, indent=2, ensure_ascii=False))
        print("\nRAW QWEN3-4B OUTPUT:")
        print(raw_output)
        clean_json = re.sub(r"```json\s*|```", "", raw_output, flags=re.IGNORECASE).strip()
        start = clean_json.find("{")
        end = clean_json.rfind("}")
        if start == -1 or end == -1 or end <= start:
            raise ValueError("No JSON object found in Qwen3-4B output")
        parsed = json.loads(clean_json[start:end + 1])
        model_results = parsed.get("line_items") if isinstance(parsed, dict) else None
        if not isinstance(model_results, list):
            raise ValueError("Qwen3-4B output does not contain a line_items array")
        # Enforce one output per input line and preserve source order.
        normalized_results = []
        for index, item in enumerate(line_items, 1):
            model_item = model_results[index - 1] if index - 1 < len(model_results) else {}
            if not isinstance(model_item, dict):
                model_item = {}
            description = str(
                item.get("description") or
                item.get("product_name") or
                item.get("name") or
                model_item.get("source_description") or ""
            ).strip()
            tax_snapshot = _tax_snapshot(item)
            matched_id = str(model_item.get("account_id") or "").strip()
            matched_name = str(model_item.get("account_name") or "").strip()
            if matched_id in accounts_map:
                valid_id = matched_id
                valid_name = accounts_map[matched_id]
            elif matched_name.lower() in accounts_map:
                valid_id, valid_name = accounts_map[matched_name.lower()]
            else:
                valid_id = None
                valid_name = None
            try:
                confidence = float(model_item.get("confidence_score") or 0.0)
            except (TypeError, ValueError):
                confidence = 0.0
            confidence = max(0.0, min(1.0, confidence))
            model_tax = model_item.get("tax_analysis")
            if not isinstance(model_tax, dict):
                model_tax = {}
            # Start from deterministic VLM extraction and only add validated
            # Stage-2 metadata. This prevents the LLM from changing extracted
            # tax amounts/rates.
            tax_result = dict(tax_snapshot)
            tax_result["taxable_amount"] = item.get("taxable_amount")
            if available_taxes:
                zoho_tax_id = model_tax.get("zoho_tax_id")
                zoho_tax_name = model_tax.get("zoho_tax_name")
                valid_tax = None
                for tax in available_taxes:
                    tid = str(tax.get("tax_id") or tax.get("id") or "").strip()
                    tname = str(tax.get("tax_name") or tax.get("name") or "").strip()
                    if (zoho_tax_id and tid == str(zoho_tax_id)) or (
                        zoho_tax_name and tname.lower() == str(zoho_tax_name).lower()
                    ):
                        valid_tax = tax
                        break
                if valid_tax:
                    tax_result["zoho_tax_id"] = str(valid_tax.get("tax_id") or valid_tax.get("id"))
                    tax_result["zoho_tax_name"] = str(valid_tax.get("tax_name") or valid_tax.get("name") or "")
                    tax_result["zoho_tax_rate"] = _as_float(valid_tax.get("tax_rate", valid_tax.get("rate")))
                elif tax_snapshot["tax_present"]:
                    tax_result["tax_needs_review"] = True
            try:
                model_tax_conf = float(model_tax.get("tax_confidence", tax_snapshot["tax_confidence"]))
            except (TypeError, ValueError):
                model_tax_conf = tax_snapshot["tax_confidence"]
            tax_result["tax_confidence"] = round(
                max(0.0, min(1.0, min(model_tax_conf, tax_snapshot["tax_confidence"]))),
                2,
            )
            if model_tax.get("tax_needs_review") is True:
                tax_result["tax_needs_review"] = True
            needs_review = (
                valid_id is None or
                confidence < 0.70 or
                bool(tax_result.get("tax_needs_review"))
            )
            normalized_results.append({
                "line_index": index,
                "source_description": description,
                "ai_account_id": valid_id,
                "ai_account_name": valid_name,
                "ai_confidence": round(confidence, 2),
                "ai_needs_review": needs_review,
                "final_account_id": valid_id,
                "final_account_name": valid_name,
                "tax_analysis": tax_result,
            })
        # A mismatch in output length is a review condition, but we still keep
        # deterministic placeholders so Stage 3 can merge one-to-one.
        if len(model_results) != len(line_items):
            for r in normalized_results:
                r["ai_needs_review"] = True
            print(
                f"[Stage 2] WARNING: model returned {len(model_results)} line results "
                f"for {len(line_items)} input lines. Missing/extra results require review."
            )
        return normalized_results
    except Exception as e:
        print(f"[categorize_line_items] Whole-invoice Stage-2 error: {e}")
        fallback = []
        for index, item in enumerate(line_items, 1):
            snapshot = _tax_snapshot(item if isinstance(item, dict) else {})
            snapshot["taxable_amount"] = item.get("taxable_amount") if isinstance(item, dict) else None
            snapshot["tax_needs_review"] = True
            fallback.append({
                "line_index": index,
                "source_description": (item or {}).get("description", "") if isinstance(item, dict) else "",
                "ai_account_id": None,
                "ai_account_name": None,
                "ai_confidence": 0.0,
                "ai_needs_review": True,
                "final_account_id": None,
                "final_account_name": None,
                "tax_analysis": {**snapshot, "error": str(e)},
            })
        return fallback

def run_tds_stage(
    complete_invoice_json: dict,
    text_model_override=None,
    tokenizer_override=None,
) -> dict:
    """
    Stage 3 — Qwen3-4B TDS classification + calculation.

    The entire normalized invoice JSON is sent to the text model. No selected
    fields are constructed. The model may classify a TDS provision and propose
    a rate/base only when supported by the invoice/context. The final numeric
    TDS amount is calculated deterministically from the returned base and rate.
    Missing legal inputs are left null and flagged for review rather than
    invented.
    """
    if not isinstance(complete_invoice_json, dict):
        raise TypeError("complete_invoice_json must be a dictionary")

    model = text_model_override or text_model
    tokenizer = tokenizer_override or text_tokenizer

    system_prompt = """
You are Stage 3 of an Indian invoice accounting pipeline.

You receive the COMPLETE normalized invoice JSON from earlier stages.
Do not reduce it to selected fields. Read the entire JSON, including
additional_fields, tax_details, line_items, vendor/customer data, HSN/SAC,
amounts and any extracted TDS/TCS labels.

Your task is ONLY to analyze TDS (withholding tax) and return valid JSON.

Rules:
1. TDS is separate from GST. Never include CGST/SGST/IGST as TDS.
2. If the invoice explicitly prints TDS, TDS rate, TDS section, TDS base,
   deduction or a withholding amount, preserve those values and distinguish
   "extracted" from "calculated".
3. If TDS is not explicitly printed, you may classify whether TDS appears
   potentially applicable from the invoice's nature of payment, vendor,
   HSN/SAC, description and available context, but do not fabricate a
   section/rate.
4. Only provide a TDS rate when it is explicitly printed or when the current
   invoice/context makes the inference sufficiently defensible. Mark
   rate_source as "extracted", "inferred", or "unknown".
5. Determine the proposed TDS base amount explicitly. Do not silently use
   total_amount if another base is more appropriate. Prefer an invoice
   value that is actually supported by the document.
6. If a numeric TDS rate and base are available, calculate
   calculated_tds_amount = base * rate / 100.
7. If the invoice already prints a TDS amount, preserve it as
   extracted_tds_amount and compare it with calculated_tds_amount when both
   exist.
8. Do not invent threshold rules, PAN exceptions, treaty rules, surcharge,
   cess, or section rates that are not supported by the available data.
9. Return null for unavailable values and set needs_review=true when legal
   classification or the rate/base is uncertain.

Return ONLY:
{
  "applicable": true/false/null,
  "tds_type": null,
  "nature_of_payment": null,
  "tds_provision": null,
  "tds_section": null,
  "tds_rate": null,
  "rate_source": "extracted|inferred|unknown",
  "tds_base_amount": null,
  "base_source": "extracted|derived|unknown",
  "extracted_tds_amount": null,
  "calculated_tds_amount": null,
  "calculation": null,
  "confidence": 0.0,
  "needs_review": true/false,
  "reason": null
}
"""

    payload = {
        "complete_normalized_invoice_json": copy.deepcopy(complete_invoice_json)
    }
    messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": (
                "Analyze the COMPLETE invoice JSON below. Do not omit "
                "additional_fields or tax metadata.\n\n" +
                json.dumps(payload, ensure_ascii=False, indent=2)
            ),
        },
    ]

    try:
        text_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        inputs = tokenizer([text_prompt], return_tensors="pt").to(model.device)
        pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id
        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=768,
                do_sample=False,
                pad_token_id=pad_id,
            )
        generated = outputs[0][inputs.input_ids.shape[1]:]
        raw = tokenizer.decode(generated, skip_special_tokens=True).strip()
        parsed = safe_json_parse(raw)

        if not isinstance(parsed, dict):
            raise ValueError("Stage 3 TDS output is not a JSON object")

        # Keep deterministic numeric cleaning.
        for key in ("tds_rate", "tds_base_amount", "extracted_tds_amount", "calculated_tds_amount"):
            if key in parsed:
                parsed[key] = _clean_numeric(parsed.get(key))

        rate = _as_float(parsed.get("tds_rate"))
        base = _as_float(parsed.get("tds_base_amount"))
        extracted = _as_float(parsed.get("extracted_tds_amount"))

        # Deterministic calculation ONLY when rate + base are available.
        deterministic_amount = None
        if rate is not None and base is not None:
            deterministic_amount = round(base * rate / 100.0, 2)
            parsed["calculated_tds_amount"] = deterministic_amount
            parsed["calculation"] = f"{base:.2f} × {rate:.4f}% = {deterministic_amount:.2f}"

        if extracted is not None and deterministic_amount is not None:
            parsed["extracted_vs_calculated_difference"] = round(
                extracted - deterministic_amount, 2
            )

        # If no usable rate/base exists, do not manufacture a TDS amount.
        if deterministic_amount is None:
            parsed["calculated_tds_amount"] = None

        confidence = _as_float(parsed.get("confidence")) or 0.0
        parsed["confidence"] = round(max(0.0, min(1.0, confidence)), 2)

        # Legal/accounting uncertainty must remain reviewable.
        parsed["needs_review"] = bool(
            parsed.get("needs_review", False) or
            (parsed.get("applicable") is True and (rate is None or base is None))
        )
        parsed["raw_model_output"] = raw
        return parsed

    except Exception as exc:
        return {
            "applicable": None,
            "tds_type": None,
            "nature_of_payment": None,
            "tds_provision": None,
            "tds_section": None,
            "tds_rate": None,
            "rate_source": "unknown",
            "tds_base_amount": None,
            "base_source": "unknown",
            "extracted_tds_amount": None,
            "calculated_tds_amount": None,
            "calculation": None,
            "confidence": 0.0,
            "needs_review": True,
            "reason": f"stage3_error:{type(exc).__name__}: {exc}",
        }


def classify_invoice(
    o1: dict,
    chart_of_accounts: list = None,
    available_taxes: list = None,
) -> tuple[dict, str]:
    """Stage 2: send the complete Stage-1 `o1` JSON to Qwen3-4B in one call."""
    if not isinstance(o1, dict):
        raise TypeError("Stage-1 o1 must be a dictionary")

    cat_results = categorize_line_items(
        line_items=o1.get("line_items") or [],
        chart_of_accounts=chart_of_accounts or [],
        vendor_name=o1.get("vendor_name") or "",
        invoice_context=_invoice_tax_context(o1),
        available_taxes=available_taxes or [],
        full_invoice_json=o1,
    )

    o2 = {
        "source": "complete_stage1_json",
        "line_items": cat_results,
    }
    return o2, json.dumps(o2, ensure_ascii=False)

def merge_invoice_results(o1: dict, o2: dict) -> dict:
    """Stage 3: merge account + tax analysis into the extracted line items."""
    if not isinstance(o1, dict) or not isinstance(o2, dict):
        raise TypeError("o1 and o2 must both be dictionaries")

    final = copy.deepcopy(o1)
    cat_items = o2.get("line_items") or []
    orig_items = final.get("line_items") or []

    for idx, item in enumerate(orig_items):
        if idx < len(cat_items) and isinstance(cat_items[idx], dict):
            item.update(cat_items[idx])

    # Preserve an invoice-level tax summary as well, without overwriting
    # the VLM-extracted source fields.
    final = ensure_lossless_tax_metadata(final)
    final["tax_analysis"] = {
        "invoice_tax_total_extracted": final.get("tax_total"),
        "line_tax_totals": _invoice_tax_context(final).get("line_tax_totals"),
        "observed_tax_rates": _invoice_tax_context(final).get("observed_tax_rates"),
        "tax_details": copy.deepcopy(
            (final.get("additional_fields") or {}).get("tax_details") or {}
        ),
    }

    return final


def run_two_stage_invoice(
    image_path: str,
    chart_of_accounts: list = None,
    available_taxes: list = None,
) -> dict:
    """Complete pipeline used by the backend.

    Stage 1 = Qwen3-VL invoice extraction/validation.
    Stage 2 = Qwen3-4B receives the COMPLETE Stage-1 JSON in one call and performs
              accounting categorization + GST/tax analysis using live Zoho data.
    Stage 3 = merge.
    """
    stage1 = extract_invoice(image_path)
    o1 = stage1.get("data")

    if not o1:
        return {
            **stage1,
            "o2": None,
            "final": None,
            "agentic_stage_2_error": "Stage 1 produced no usable invoice data",
        }

    try:
        o2, accounting_raw = classify_invoice(
            o1,
            chart_of_accounts=chart_of_accounts,
            available_taxes=available_taxes,
        )
        final = merge_invoice_results(o1, o2)

        # Stage 3 receives the COMPLETE normalized JSON, not selected fields.
        tds_analysis = run_tds_stage(final)
        final["tds_analysis"] = tds_analysis

        # Keep TDS duplicated in additional_fields for lossless downstream
        # consumers that already read the metadata object.
        extras = final.setdefault("additional_fields", {})
        if not isinstance(extras, dict):
            extras = {}
            final["additional_fields"] = extras
        extras["tds_analysis"] = copy.deepcopy(tds_analysis)

        return {
            **stage1,
            "o1": o1,
            "o2": o2,
            "o2_raw": accounting_raw,
            "o3_tds": tds_analysis,
            "final": final,
        }
    except Exception as e:
        review_reasons = list(stage1.get("review_reasons", []))
        review_reasons.append(f"stage_2_accounting_error:{type(e).__name__}:{e}")
        return {
            **stage1,
            "o1": o1,
            "o2": None,
            "o2_raw": None,
            "final": o1,
            "needs_review": True,
            "review_reasons": review_reasons,
        }

## 7. JSON parsing + cleaning/normalization (fallback path only)

In [9]:
_FENCE_RE = re.compile(r"^```(?:json)?\s*|\s*```$", re.MULTILINE)

def strip_code_fences(text):
    return _FENCE_RE.sub("", text).strip()

def extract_json_block(text):
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return text[start:i + 1]
    return None

def repair_truncated_json(text):
    """Best-effort repair for JSON truncated mid-generation:
    closes an open string, then closes any open brackets/braces."""
    s = text.strip()
    if not s.startswith("{"):
        start = s.find("{")
        if start == -1:
            return None
        s = s[start:]

    def scan(s):
        stack = []
        in_string = False
        escape = False
        for ch in s:
            if in_string:
                if escape:
                    escape = False
                elif ch == "\\":
                    escape = True
                elif ch == '"':
                    in_string = False
                continue
            if ch == '"':
                in_string = True
            elif ch in "{[":
                stack.append(ch)
            elif ch in "}]":
                if stack:
                    stack.pop()
        return stack, in_string

    stack, in_string = scan(s)

    # If we ended mid-string, trim back to before the dangling key/value
    # and recompute the bracket stack on the trimmed text.
    if in_string:
        trim_at = max(s.rfind(",", 0, len(s)), s.rfind("{", 0, len(s)))
        s = s[:trim_at] if trim_at != -1 else s
        stack, in_string = scan(s)

    s = re.sub(r",\s*$", "", s)  # strip trailing comma before closing
    closers = {"{": "}", "[": "]"}
    s += "".join(closers[c] for c in reversed(stack))
    return s

def safe_json_parse(raw_text):
    cleaned = strip_code_fences(raw_text)
    for candidate in (cleaned, extract_json_block(cleaned)):
        if not candidate:
            continue
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            continue

    # NEW: last-resort attempt to repair truncated/malformed JSON
    repaired = repair_truncated_json(cleaned)
    if repaired:
        try:
            return json.loads(repaired)
        except json.JSONDecodeError:
            pass

    return None

NUMERIC_TOP_FIELDS = {"subtotal", "discount_total", "tax_total", "shipping_charges",
                       "other_charges", "round_off", "total_amount"}
NUMERIC_LINE_ITEM_FIELDS = {"quantity", "unit_price", "discount",
                             "taxable_amount",
                             "cgst_rate", "cgst_amount",
                             "sgst_rate", "sgst_amount",
                             "igst_rate", "igst_amount",
                             "total"}

def _clean_numeric(v):
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return float(v)
    if isinstance(v, str):
        s = v.strip()
        # Accounting convention: "(1,234.56)" means -1234.56.
        negative = s.startswith("(") and s.endswith(")")
        # FIX: Indian invoices commonly write amounts as "Rs.5000/-" where
        # "/-" means "only" (not a minus sign) -- strip that idiom first so
        # the trailing "-" isn't later mistaken for a negative sign.
        s = re.sub(r"/[-=]\s*$", "", s)
        # FIX: strip currency words/symbols (and any other letters) BEFORE the
        # digit/dot filter. Previously "Rs. 30,134.00" kept the period in
        # "Rs." (since "." passed the old filter untouched), producing the
        # unparseable string ".30134.00" and silently returning None for a
        # perfectly good number -- a real risk on Indian invoices where
        # "Rs." is a very common prefix.
        s = re.sub(r"[A-Za-z₹$€£]+\.?", "", s)
        cleaned = re.sub(r"[^\d.\-]", "", s)
        # FIX: collapse multiple decimal points -- keep only the last as the
        # true decimal separator (any earlier ones are noise, e.g. leftover
        # currency-abbreviation punctuation) instead of failing to parse.
        if cleaned.count(".") > 1:
            head, _, tail = cleaned.rpartition(".")
            cleaned = head.replace(".", "") + "." + tail
        if cleaned in ("", "-", ".", "-."):
            return None
        try:
            val = float(cleaned)
        except ValueError:
            return None
        return -abs(val) if negative else val
    return None

def _norm_key(k):
    return re.sub(r"[^a-z0-9]+", " ", str(k).lower()).strip()

# A bare "Discount" column header is far more commonly an absolute currency
# amount than a percentage on real invoices, so "discount"/"disc" map to the
# amount field `discount`, not a percentage field.
# Note "cgst"/"sgst"/"igst" bare aliases map to the *amount* fields --
# "cgst rate"/"cgst %"/"cgst pct" map separately to the *rate* fields below,
# so a table with both an amount column and a rate column keeps them apart.
LINE_ITEM_KEY_ALIASES = {
    "description": {"description", "product name", "product name description", "item",
                     "item description", "particulars", "product"},
    "hsn_code": {"hsn code", "hsn", "hsn sac", "sac"},
    "quantity": {"qty", "qty nos", "quantity", "nos", "pcs"},
    "unit_price": {"unit price", "rate", "price", "unit rate"},
    "discount": {"disc amt", "discount amt", "discount amount", "disc value", "discount"},
    "taxable_amount": {"taxable amt rs", "taxable amount", "taxable amt", "taxable value", "assessable value"},
    "cgst_rate": {"cgst rate", "cgst %", "cgst pct", "cgst percent"},
    "cgst_amount": {"cgst rs", "cgst", "cgst amt"},
    "sgst_rate": {"sgst rate", "sgst %", "sgst pct", "sgst percent"},
    "sgst_amount": {"sgst rs", "sgst", "sgst amt"},
    "igst_rate": {"igst rate", "igst %", "igst pct", "igst percent"},
    "igst_amount": {"igst rs", "igst", "igst amt"},
    "total": {"total rs", "total", "amount total", "line total", "amount", "item total"},
}
_ALIAS_LOOKUP = {alias: canon for canon, aliases in LINE_ITEM_KEY_ALIASES.items() for alias in aliases}

def normalize_line_item(item):
    if not isinstance(item, dict):
        return {}
    out = {}
    stray = {}
    for k, v in item.items():
        canon = _ALIAS_LOOKUP.get(_norm_key(k))
        if canon is None:
            if str(k) != "additional_fields":
                stray[str(k)] = v
            continue
        out[canon] = _clean_numeric(v) if canon in NUMERIC_LINE_ITEM_FIELDS else v

    # Never drop row-level information that has no canonical field.
    existing_extra = item.get("additional_fields")
    if isinstance(existing_extra, dict):
        stray = {**existing_extra, **stray}

    if stray:
        out["additional_fields"] = stray
    return out


def loosely_validate(data):
    """Repairs the free-form JSON dict returned by the model into a valid
    InvoiceSchema instance: cleans numeric fields, drops/quarantines fields
    that don't validate (into additional_fields) instead of failing the
    whole record."""
    if not isinstance(data, dict):
        return None

    working = dict(data)
    if isinstance(working.get("line_items"), list):
        working["line_items"] = [normalize_line_item(li) for li in working["line_items"] if isinstance(li, dict)]
    for f in NUMERIC_TOP_FIELDS:
        if f in working:
            working[f] = _clean_numeric(working[f])

    filtered = {k: v for k, v in working.items() if k in InvoiceSchema.model_fields}
    stray = {k: v for k, v in working.items() if k not in InvoiceSchema.model_fields}
    if stray:
        filtered.setdefault("additional_fields", {})
        if isinstance(filtered["additional_fields"], dict):
            filtered["additional_fields"].update(stray)

    for _ in range(len(InvoiceSchema.model_fields) + 2):
        try:
            return InvoiceSchema(**filtered).model_dump()
        except ValidationError as e:
            bad_fields = {err["loc"][0] for err in e.errors() if err.get("loc")}
            if not bad_fields:
                return None
            progressed = False
            for bf in bad_fields:
                if bf == "line_items":
                    kept = []
                    for li in filtered.get("line_items", []):
                        try:
                            kept.append(LineItem(**li).model_dump())
                        except ValidationError:
                            pass
                    if len(kept) != len(filtered.get("line_items", [])):
                        filtered["line_items"] = kept
                        progressed = True
                elif bf in filtered:
                    val = filtered.pop(bf)
                    filtered.setdefault("additional_fields", {})
                    if isinstance(filtered["additional_fields"], dict):
                        filtered["additional_fields"][f"unparsed_{bf}"] = val
                    progressed = True
            if not progressed:
                return None
    return None


def loosely_validate_header(data):
    """Same repair-and-drop strategy as loosely_validate, but against
    InvoiceHeaderSchema (no line_items) -- used for the header generation
    call now that both calls are free-form."""
    if not isinstance(data, dict):
        return None

    working = dict(data)
    for f in NUMERIC_TOP_FIELDS:
        if f in working:
            working[f] = _clean_numeric(working[f])

    filtered = {k: v for k, v in working.items() if k in InvoiceHeaderSchema.model_fields}
    stray = {k: v for k, v in working.items() if k not in InvoiceHeaderSchema.model_fields}
    if stray:
        filtered.setdefault("additional_fields", {})
        if isinstance(filtered["additional_fields"], dict):
            filtered["additional_fields"].update(stray)

    for _ in range(len(InvoiceHeaderSchema.model_fields) + 2):
        try:
            return InvoiceHeaderSchema(**filtered).model_dump()
        except ValidationError as e:
            bad_fields = {err["loc"][0] for err in e.errors() if err.get("loc")}
            if not bad_fields:
                return None
            progressed = False
            for bf in bad_fields:
                if bf in filtered:
                    val = filtered.pop(bf)
                    filtered.setdefault("additional_fields", {})
                    if isinstance(filtered["additional_fields"], dict):
                        filtered["additional_fields"][f"unparsed_{bf}"] = val
                    progressed = True
            if not progressed:
                return None
    return None

## 8. Deterministic validation — regex, checksums, cross-derivation (always runs)

In [10]:
from datetime import timedelta

CODE36 = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"

PATTERNS = {
    "vendor_gstin": re.compile(r"\b\d{2}[A-Z]{5}\d{4}[A-Z][1-9A-Z]Z[0-9A-Z]\b"),
    "customer_gstin": re.compile(r"\b\d{2}[A-Z]{5}\d{4}[A-Z][1-9A-Z]Z[0-9A-Z]\b"),
    "vendor_pan": re.compile(r"\b[A-Z]{5}\d{4}[A-Z]\b"),
    "customer_pan": re.compile(r"\b[A-Z]{5}\d{4}[A-Z]\b"),
    "vendor_cin": re.compile(r"\b[UL]\d{5}[A-Z]{2}\d{4}[A-Z]{3}\d{6}\b"),
    "vendor_phone": re.compile(r"\b(?:\+?91[\-\s]?)?[6-9]\d{9}\b"),
    "vendor_email": re.compile(r"[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}"),
    "invoice_date": re.compile(r"\b\d{1,2}[\/\-\.]\d{1,2}[\/\-\.]\d{2,4}\b|\b\d{4}[\/\-]\d{1,2}[\/\-]\d{1,2}\b"),
    "due_date": re.compile(r"\b\d{1,2}[\/\-\.]\d{1,2}[\/\-\.]\d{2,4}\b|\b\d{4}[\/\-]\d{1,2}[\/\-]\d{1,2}\b"),
    # Bank sub-fields -- only used to fill a gap the model left null, never
    # to override an extracted value (see fill_missing_bank_fields).
    "ifsc_code": re.compile(r"\b[A-Z]{4}0[A-Z0-9]{6}\b"),
    "account_number": re.compile(r"\b\d{9,18}\b"),
    "upi_id": re.compile(r"\b[a-zA-Z0-9.\-_]{2,64}@[a-zA-Z]{2,64}\b"),
}

LABEL_HINTS = {
    "vendor_gstin": ["gstin", "gst no", "gst reg", "gst number", "gst in", "tax registration"],
    "customer_gstin": ["gstin", "gst no", "gst reg", "gst number"],
    "vendor_pan": ["pan no", "pan number", "pan:", "permanent account number"],
    "customer_pan": ["pan no", "pan number", "pan:", "permanent account number"],
    "vendor_cin": ["cin", "cin no", "corporate identity"],
    "vendor_phone": ["phone", "mobile", "contact", "tel", "ph."],
    "vendor_email": ["email", "e-mail", "mail id"],
    "invoice_date": ["invoice date", "date", "dated", "bill date", "document date", "issue date"],
    "due_date": ["due date", "payment due", "payable by", "pay by"],
    "ifsc_code": ["ifsc", "ifsc code", "ifs code"],
    "account_number": ["a/c no", "account no", "account number", "bank a/c", "acc no", "a c no"],
    "upi_id": ["upi", "upi id", "vpa"],
}

def is_valid_gstin_format(v):
    return bool(v) and bool(PATTERNS["vendor_gstin"].fullmatch(v.strip().upper()))

def is_valid_gstin_checksum(v):
    # Verified against two independently-published GSTIN Luhn mod-36 worked
    # examples (27AABCU9603R1ZN and 27AAPFU0939F1ZV) -- this formula is correct
    # as originally written. Left unchanged; do not "fix" this again without
    # re-verifying against a real worked example first.
    if not v or len(v) != 15:
        return False
    v = v.strip().upper()
    try:
        factor = 1
        total = 0
        for ch in v[:-1]:
            digit = CODE36.index(ch)
            code_point = factor * digit
            factor = 2 if factor == 1 else 1
            code_point = (code_point // 36) + (code_point % 36)
            total += code_point
        check_digit = CODE36[(36 - (total % 36)) % 36]
        return check_digit == v[-1]
    except (ValueError, IndexError):
        return False

def is_valid_pan_format(v):
    return bool(v) and bool(re.fullmatch(r"[A-Z]{5}\d{4}[A-Z]", v.strip().upper()))

def is_valid_ifsc_format(v):
    return bool(v) and bool(re.fullmatch(r"[A-Z]{4}0[A-Z0-9]{6}", v.strip().upper()))

# Only ifsc_code/account_number/upi_id have patterns reliable enough to
# regex-match on their own; bank_name/branch/account_holder_name are left
# exactly as the model returned them (see fill_missing_bank_fields).
BANK_FIELD_VALIDATORS = {
    "ifsc_code": is_valid_ifsc_format,
    "account_number": lambda v: bool(v) and v.strip().isdigit() and 9 <= len(v.strip()) <= 18,
    "upi_id": lambda v: bool(v) and bool(PATTERNS["upi_id"].fullmatch(v.strip())),
}

def derive_pan_from_gstin(gstin):
    if gstin and len(gstin) == 15:
        candidate = gstin[2:12].upper()
        if is_valid_pan_format(candidate):
            return candidate
    return None

def normalize_date(value):
    if not value:
        return None
    try:
        dt = dateparser.parse(value, dayfirst=True, fuzzy=True)
        return dt.strftime("%Y-%m-%d")
    except (ValueError, OverflowError, TypeError):
        return value

# "Net 30", "30 days from invoice date", "Due on receipt", "50% advance", etc.
# Deliberately simple -- anything it can't parse just leaves due_date null,
# same as before; it never guesses.
_NET_TERMS_RE = re.compile(r"net\s*(\d{1,3})|(\d{1,3})\s*days?", re.IGNORECASE)
_IMMEDIATE_TERMS = ("due on receipt", "immediate", "advance", "cash", "cod", "payable on receipt", "prepaid")

def compute_due_date_from_terms(payment_terms, invoice_date):
    """Best-effort due date from payment_terms text + invoice_date. Only ever
    used to FILL a missing due_date -- never overrides a due_date the model
    actually read off the invoice (see extract_invoice).

    NOTE: by the time this runs, invoice_date has already been through
    normalize_date() and is YYYY-MM-DD -- parse it WITHOUT dayfirst=True, or
    an unambiguous ISO date like 2026-07-01 gets misread as day=07 (dayfirst
    forces day-month-year even on year-first strings)."""
    if not payment_terms or not invoice_date:
        return None
    try:
        base = dateparser.parse(invoice_date)
    except (ValueError, OverflowError, TypeError):
        return None
    terms_lower = payment_terms.lower()
    if any(term in terms_lower for term in _IMMEDIATE_TERMS):
        return base.strftime("%Y-%m-%d")
    m = _NET_TERMS_RE.search(terms_lower)
    if m:
        days = int(m.group(1) or m.group(2))
        return (base + timedelta(days=days)).strftime("%Y-%m-%d")
    return None

def regex_fallback(field, raw_text, start_after=0):
    """Returns (matched_value_or_None, match_start_index_or_-1).

    Accepts `start_after` so a caller can force a second, related lookup
    (e.g. customer_gstin after vendor_gstin) to search past the first match
    instead of re-finding the same label/window -- otherwise vendor_gstin and
    customer_gstin can both resolve to the same text since they share label
    keywords like "gstin".
    """
    pattern = PATTERNS.get(field)
    if pattern is None or not raw_text:
        return None, -1
    lowered = raw_text.lower()
    for hint in LABEL_HINTS.get(field, []):
        idx = lowered.find(hint, start_after)
        if idx != -1:
            window = raw_text[max(0, idx - 10): idx + 120]
            m = pattern.search(window)
            if m:
                return m.group().strip(), idx
    m = pattern.search(raw_text, start_after)
    return (m.group().strip(), m.start()) if m else (None, -1)

TRANSCRIBE_PROMPT = (
    "Transcribe every piece of text visible in this image exactly as printed, "
    "line by line, including all labels, numbers, codes, and dates. Do not "
    "summarize, interpret, or omit anything. Plain text only -- no JSON, no "
    "commentary."
)

def _build_transcribe_messages(image_path):
    return [
        {"role": "system", "content": "You transcribe text from images verbatim."},
        {"role": "user", "content": [
            {"type": "image", "image": image_path, "min_pixels": MIN_PIXELS, "max_pixels": MAX_PIXELS},
            {"type": "text", "text": TRANSCRIBE_PROMPT},
        ]},
    ]

def get_backup_text_via_reread(image_path):
    """Re-reads the invoice image with the loaded VLM, asking for a raw
    transcription. Used as the fallback text source for regex_fallback()
    wherever a field fails validation. Wrapped in try/except because this is
    already a fallback path -- a second failure here should degrade to an
    empty string, not crash the pipeline."""
    try:
        return _run_generation(image_path, _build_transcribe_messages, max_new_tokens=1024)
    except Exception:
        return ""

## 9. Arithmetic reconciliation

Additive, not multiplicative -- `total ≠ tax × subtotal`. Discount is applied first
(percentage and/or flat amount), tax is added on top of the discounted base, and
`other_charges`/`shipping_charges`/`round_off` are separate additive terms at the
invoice level.

In [11]:
def reconcile_line_item(item, tolerance=0.02):
    qty, price = item.get("quantity"), item.get("unit_price")
    stated_total = item.get("total")

    if qty is None or price is None or stated_total is None:
        return {"status": "incomplete"}

    gross = qty * price
    discount_amt = item.get("discount") or 0
    taxable_declared = item.get("taxable_amount")
    cgst = item.get("cgst_amount")
    sgst = item.get("sgst_amount")
    igst = item.get("igst_amount")

    taxable_computed = gross - discount_amt
    base = taxable_declared if taxable_declared is not None else taxable_computed

    # Per-tax-type rates (cgst_rate/sgst_rate/igst_rate), summed -- an extra
    # arithmetic-check candidate alongside the amount-based one below.
    rate_parts = [item.get("cgst_rate"), item.get("sgst_rate"), item.get("igst_rate")]
    split_rate = sum(r for r in rate_parts if r is not None) if any(r is not None for r in rate_parts) else None

    has_tax_info = any(v is not None for v in (cgst, sgst, igst)) or split_rate is not None

    if not has_tax_info:
        if abs(gross - stated_total) <= tolerance:
            return {"status": "ok", "matched_formula": "gross_equals_total_no_tax"}
        if taxable_declared is not None and abs(taxable_declared - stated_total) <= tolerance:
            return {"status": "ok", "matched_formula": "taxable_equals_total_no_tax"}
        if stated_total > base + tolerance:
            return {
                "status": "unverifiable_tax_breakdown",
                "pretax_amount": round(base, 2), "stated_total": stated_total,
                "implied_tax": round(stated_total - base, 2),
            }
        return {"status": "mismatch", "candidates": {"gross_only": round(gross, 2)}, "stated": stated_total}

    tax_total = sum(v for v in (cgst, sgst, igst) if v is not None)

    candidates = {"taxable_plus_tax": base + tax_total}
    if split_rate is not None:
        candidates["taxable_times_split_rates"] = base * (1 + split_rate / 100)

    for label, val in candidates.items():
        if abs(val - stated_total) <= tolerance:
            return {"status": "ok", "matched_formula": label, "computed": round(val, 2)}

    return {"status": "mismatch", "candidates": {k: round(v, 2) for k, v in candidates.items()}, "stated": stated_total}


def reconcile_invoice(data, tolerance=0.5):
    line_items = data.get("line_items") or []
    line_total_sum = sum(li.get("total") for li in line_items if isinstance(li.get("total"), (int, float)))

    # Only sum taxable_amount when EVERY line item has it -- falling back to
    # `total` (post-tax) per-item when taxable_amount is missing would mix
    # pre-tax and post-tax bases into one "taxable" sum, which isn't
    # apples-to-apples.
    taxable_vals = [li.get("taxable_amount") for li in line_items if isinstance(li.get("taxable_amount"), (int, float))]
    taxable_basis_complete = len(line_items) > 0 and len(taxable_vals) == len(line_items)
    line_taxable_sum = sum(taxable_vals) if taxable_vals else None

    subtotal = data.get("subtotal")
    discount_total = data.get("discount_total") or 0
    tax_total = data.get("tax_total") or 0
    shipping = data.get("shipping_charges") or 0
    other_charges = data.get("other_charges") or 0
    round_off = data.get("round_off") or 0
    total_amount = data.get("total_amount")

    result = {"line_items_sum": round(line_total_sum, 2) if line_items else None}

    if subtotal is not None and line_items:
        if taxable_basis_complete:
            result["subtotal_match"] = abs(line_taxable_sum - subtotal) <= tolerance
        else:
            result["subtotal_match"] = None
            result["subtotal_check_note"] = "incomplete_taxable_amount_on_line_items"

    if total_amount is not None:
        # subtotal is defined as the post-discount taxable value in the
        # prompt ("Taxable Amount"/"Taxable Value"/"Assessable Value"), so it
        # is already net of discount and discount_total is NOT subtracted a
        # second time here.
        base = subtotal if subtotal is not None else (line_taxable_sum if taxable_basis_complete else line_total_sum)
        reconstructed = base + tax_total + shipping + other_charges + round_off
        result["total_reconstructed"] = round(reconstructed, 2)
        result["total_match"] = abs(reconstructed - total_amount) <= tolerance
        if discount_total:
            result["total_reconciliation_note"] = (
                "discount_total not subtracted here -- subtotal is assumed already "
                "net of discount per the extraction prompt's field definitions"
            )

    return result

## 10. Master single-invoice pipeline

No batching, no self-consistency across multiple samples, no evaluation report --
one image in, one annotated JSON result out. Corrections (regex fallback,
GSTIN-derived PAN, computed due date, bank sub-field fill-in) only ever kick
in when the model failed to produce a valid value for that specific field --
an already-extracted value is never second-guessed or overwritten.

In [ ]:
STRUCTURED_VALIDATORS = {
    "vendor_gstin": lambda v: is_valid_gstin_format(v) and is_valid_gstin_checksum(v),
    "customer_gstin": lambda v: is_valid_gstin_format(v) and is_valid_gstin_checksum(v),
    "vendor_pan": is_valid_pan_format,
    "customer_pan": is_valid_pan_format,
    "vendor_cin": lambda v: bool(v) and bool(PATTERNS["vendor_cin"].fullmatch(v.strip().upper())),
    "vendor_phone": lambda v: bool(v) and bool(PATTERNS["vendor_phone"].fullmatch(v.strip())),
    "vendor_email": lambda v: bool(v) and bool(PATTERNS["vendor_email"].fullmatch(v.strip())),
}

# Fields whose total absence should always route to human review, even if
# every other check passes -- these are the ones an AP workflow can't
# function without.
REQUIRED_IDENTITY_FIELDS = ["vendor_name", "customer_name", "invoice_number", "total_amount"]


def rescue_via_reread(image_path: str) -> dict:
    """Last-resort extraction when the model produced nothing usable at all
    (raised, or returned output that couldn't be parsed/validated). Re-reads
    the image with the VLM for a plain transcription and regex-matches every
    field we have a pattern for, so a total model failure still yields a
    partial, reviewable record instead of an empty one."""
    backup_text = get_backup_text_via_reread(image_path)
    data, field_sources = {}, {}
    gstin_match_pos = 0
    for field in PATTERNS:
        search_from = gstin_match_pos if field == "customer_gstin" else 0
        value, pos = regex_fallback(field, backup_text, start_after=search_from)
        if field == "vendor_gstin" and pos != -1:
            gstin_match_pos = pos + 1
        validator = STRUCTURED_VALIDATORS.get(field) or BANK_FIELD_VALIDATORS.get(field)
        if value and (validator is None or validator(value)):
            data[field] = value
            field_sources[field] = "reread_fallback"
    for date_field in ("invoice_date", "due_date"):
        if data.get(date_field):
            data[date_field] = normalize_date(data[date_field])
    return {"data": data, "field_sources": field_sources}


def fill_missing_bank_fields(data, get_backup_text_fn):
    """Bank sub-fields are only ever CORRECTED when the model failed to
    fetch that specific sub-field -- never overwritten if the model already
    returned a value. Only ifsc_code/account_number/upi_id have patterns
    reliable enough to regex-match; bank_name/branch/account_holder_name are
    left exactly as the model returned them."""
    bd = data.get("bank_details")
    if not isinstance(bd, dict):
        bd = {"raw_text": bd} if bd else {}

    missing = [f for f in ("ifsc_code", "account_number", "upi_id") if not bd.get(f)]
    if missing:
        backup_text = get_backup_text_fn()
        for field in missing:
            value, _ = regex_fallback(field, backup_text)
            validator = BANK_FIELD_VALIDATORS.get(field)
            if value and (validator is None or validator(value)):
                bd[field] = value

    data["bank_details"] = bd or None


def extract_invoice(image_path: str) -> dict:
    # Generation can legitimately fail (CUDA OOM, a bad frame from a
    # malformed PDF, a transient model error) -- surface a reviewable result
    # instead of crashing the run.
    try:
        data, raw_texts = generate_invoice(image_path)
    except Exception as e:
        rescue = rescue_via_reread(image_path)
        return {
            "data": rescue["data"] or None,
            "field_sources": rescue["field_sources"],
            "needs_review": True,
            "review_reasons": [f"generation_error:{type(e).__name__}: {e}", "reread_rescue_attempted"],
            "raw_output": None, "generation_path": "error_reread_rescue",
        }

    if data is None:
        rescue = rescue_via_reread(image_path)
        return {
            "data": rescue["data"] or None,
            "field_sources": rescue["field_sources"],
            "needs_review": True,
            "review_reasons": ["model_output_unparseable", "reread_rescue_attempted"],
            "raw_output": raw_texts, "generation_path": "free_form_header_plus_line_items_reread_rescue",
        }

    source_note = "free_form_header_plus_line_items"

    # Backup text (a plain re-transcription) is only ever fetched lazily, the
    # first time some field actually fails validation, and cached after that
    # so it runs at most once per invoice.
    _backup_text_cache = {}
    def get_backup_text():
        if "text" not in _backup_text_cache:
            _backup_text_cache["text"] = get_backup_text_via_reread(image_path)
        return _backup_text_cache["text"]

    review_reasons = []
    # Only mark a field "llm"-sourced if the model actually returned a
    # non-null value for it.
    field_sources = {f: "llm" for f in CORE_FIELDS if data.get(f) is not None}

    # Track where the vendor_gstin match landed on the page so the
    # customer_gstin fallback lookup (which shares label keywords like
    # "gstin") is forced to search past it instead of re-finding the same
    # text.
    gstin_match_pos = 0
    for field, validator in STRUCTURED_VALIDATORS.items():
        value = data.get(field)
        valid = bool(value) and validator(value)
        if not valid:
            search_from = gstin_match_pos if field == "customer_gstin" else 0
            fallback, pos = regex_fallback(field, get_backup_text(), start_after=search_from)
            if field == "vendor_gstin" and pos != -1:
                gstin_match_pos = pos + 1
            if fallback and validator(fallback):
                data[field] = fallback
                field_sources[field] = "reread_fallback"
            elif value:
                review_reasons.append(f"{field}_failed_validation")

    # Explicit cross-field invariant -- vendor and customer must never share
    # a GSTIN.
    if data.get("vendor_gstin") and data.get("vendor_gstin") == data.get("customer_gstin"):
        review_reasons.append("vendor_customer_gstin_identical")

    # PAN derived from GSTIN only when the model didn't already give us a
    # valid one -- vendor and customer get the same treatment.
    if not data.get("vendor_pan") or not is_valid_pan_format(data.get("vendor_pan")):
        derived_pan = derive_pan_from_gstin(data.get("vendor_gstin"))
        if derived_pan:
            data["vendor_pan"] = derived_pan
            field_sources["vendor_pan"] = "derived_from_gstin"

    if not data.get("customer_pan") or not is_valid_pan_format(data.get("customer_pan")):
        derived_customer_pan = derive_pan_from_gstin(data.get("customer_gstin"))
        if derived_customer_pan:
            data["customer_pan"] = derived_customer_pan
            field_sources["customer_pan"] = "derived_from_gstin"

    for date_field in ("invoice_date", "due_date"):
        if data.get(date_field):
            data[date_field] = normalize_date(data[date_field])

    # due_date is ONLY computed when the invoice genuinely doesn't show one --
    # never overrides a due_date the model actually read off the page.
    if not data.get("due_date"):
        computed_due = compute_due_date_from_terms(data.get("payment_terms"), data.get("invoice_date"))
        if computed_due:
            data["due_date"] = computed_due
            field_sources["due_date"] = "computed_from_payment_terms"

    # Bank sub-fields: only fills gaps, never overwrites what the model read.
    fill_missing_bank_fields(data, get_backup_text)

    line_item_checks = [reconcile_line_item(li) for li in data.get("line_items", [])]
    for i, chk in enumerate(line_item_checks):
        if chk["status"] == "mismatch":
            review_reasons.append(f"line_item_{i}_arithmetic_mismatch")
    unverifiable_count = sum(1 for c in line_item_checks if c["status"] == "unverifiable_tax_breakdown")
    if unverifiable_count:
        review_reasons.append(f"{unverifiable_count}_line_item(s)_missing_tax_breakdown")

    invoice_check = reconcile_invoice(data)
    if invoice_check.get("total_match") is False:
        review_reasons.append("invoice_total_mismatch")
    if invoice_check.get("subtotal_match") is False:
        review_reasons.append("invoice_subtotal_mismatch")

    # Basic completeness gate -- an invoice missing line items entirely, or
    # missing the fields an AP workflow can't function without, should
    # always be reviewable rather than silently passing.
    if not data.get("line_items"):
        review_reasons.append("no_line_items_extracted")
    for f in REQUIRED_IDENTITY_FIELDS:
        if not data.get(f):
            review_reasons.append(f"{f}_missing")

    return {
        "data": data,
        "field_sources": field_sources,
        "line_item_reconciliation": line_item_checks,
        "line_items_with_unverifiable_tax": unverifiable_count,
        "invoice_reconciliation": invoice_check,
        "needs_review": len(review_reasons) > 0,
        "review_reasons": review_reasons,
        "generation_path": source_note,
    }

# ============================================================
# BLOCK 10 — STAGE-2 COMPATIBILITY WRAPPER
# ============================================================
# The notebook previously contained a second, hard-coded 33-category Stage-2
# implementation here. That duplicate implementation could raise
# NameError: ZOHO_CATEGORIES and could also drop tax information.
# This block now reuses the live-Zoho, tax-aware implementation from Cell 14.


def extract_json_from_text(text):
    """Extract one JSON object from model output."""
    if not isinstance(text, str):
        raise TypeError(f"Expected string, got {type(text)}")

    cleaned = re.sub(r"```json\s*|```", "", text, flags=re.IGNORECASE).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        start = cleaned.find("{")
        end = cleaned.rfind("}")
        if start == -1 or end == -1 or end <= start:
            raise ValueError(f"No JSON object found in model output:\n{cleaned}")
        return json.loads(cleaned[start:end + 1])


def normalize_stage1_output(stage1_output):
    """Normalize the different Stage-1 return shapes used in this notebook."""
    if isinstance(stage1_output, dict):
        # If the caller already passed the full API result, prefer its data/o1.
        if isinstance(stage1_output.get("data"), dict):
            return stage1_output["data"]
        if isinstance(stage1_output.get("o1"), dict):
            return stage1_output["o1"]
        return stage1_output

    if isinstance(stage1_output, (list, tuple)):
        for item in stage1_output:
            if isinstance(item, dict):
                if "line_items" in item:
                    return item
                if isinstance(item.get("data"), dict):
                    return item["data"]
            elif isinstance(item, str):
                try:
                    parsed = extract_json_from_text(item)
                    if isinstance(parsed, dict) and "line_items" in parsed:
                        return parsed
                except Exception:
                    pass
        raise ValueError("Could not find structured invoice data inside Stage-1 output.")

    if isinstance(stage1_output, str):
        parsed = extract_json_from_text(stage1_output)
        if not isinstance(parsed, dict):
            raise ValueError("Stage-1 JSON is not an object.")
        return parsed

    raise TypeError(f"Unsupported Stage-1 output type: {type(stage1_output)}")


def categorize_invoice_lines(
    o1,
    chart_of_accounts=None,
    available_taxes=None,
):
    """Compatibility wrapper around the live-COA + tax-aware Stage-2 engine."""
    if not isinstance(o1, dict):
        raise TypeError("Expected o1 to be a dictionary.")

    return categorize_line_items(
        line_items=o1.get("line_items") or [],
        chart_of_accounts=chart_of_accounts or [],
        vendor_name=o1.get("vendor_name") or "",
        invoice_context=_invoice_tax_context(o1),
        available_taxes=available_taxes or [],
    )


def merge_line_categories(o1, line_category_results):
    """Merge Stage-2 account and tax analysis into Stage-1 line items."""
    if not isinstance(o1, dict):
        raise TypeError("o1 must be a dictionary.")
    if not isinstance(line_category_results, list):
        raise TypeError("line_category_results must be a list.")

    final_result = copy.deepcopy(o1)
    line_items = final_result.get("line_items") or []

    if len(line_items) != len(line_category_results):
        raise ValueError(
            "Number of invoice line items does not match number of "
            f"categorization results. Invoice lines: {len(line_items)}; "
            f"Results: {len(line_category_results)}"
        )

    for index, category_result in enumerate(line_category_results):
        line_items[index].update(category_result)

    final_result["line_items"] = line_items
    return final_result


def run_invoice_inference(
    image_path,
    chart_of_accounts=None,
    available_taxes=None,
):
    """Compatibility runner for the complete 3-stage pipeline."""
    result = run_two_stage_invoice(
        image_path,
        chart_of_accounts=chart_of_accounts,
        available_taxes=available_taxes,
    )
    final = result.get("final") or result.get("o1") or result.get("data")
    print("\n" + "=" * 70)
    print("RUNNING THREE-STAGE INVOICE PIPELINE")
    print("=" * 70)
    print("Stage 1 → Qwen3-VL-4B Invoice Extraction")
    print("Stage 2 → Qwen3-4B Full JSON Reconciliation / Accounting")
    print("Stage 3 → Qwen3-4B TDS Classification + Calculation")
    print("\n========== FINAL JSON ==========")
    print(json.dumps(final, indent=2, ensure_ascii=False))
    return result


# ============================================================
# BLOCK 11 — UPLOAD INVOICE → PDF TO IMAGE → INFERENCE
# ============================================================

from google.colab import files
from IPython.display import display, JSON
from PIL import Image
import os
import io

# Install PDF conversion library if needed
!pip -q install pymupdf


# ============================================================
# PDF → IMAGE FUNCTION
# ============================================================

def pdf_to_images(pdf_path, dpi=200):
    """
    Convert every PDF page into a PIL RGB image.
    """

    import fitz  # PyMuPDF

    pdf_document = fitz.open(pdf_path)

    images = []

    for page_number in range(len(pdf_document)):

        page = pdf_document[page_number]

        # Render PDF page at requested DPI
        zoom = dpi / 72

        matrix = fitz.Matrix(zoom, zoom)

        pix = page.get_pixmap(
            matrix=matrix,
            alpha=False
        )

        # Convert rendered page to PIL Image
        image_bytes = pix.tobytes("png")

        image = Image.open(
            io.BytesIO(image_bytes)
        ).convert("RGB")

        images.append(image)

    pdf_document.close()

    return images


# ============================================================
# UPLOAD
# ============================================================

print("Upload an invoice image or PDF:")

uploaded = files.upload()


# ============================================================
# PROCESS UPLOADED FILE
# ============================================================

for filename in uploaded.keys():

    print("\n" + "=" * 70)
    print(f"Processing: {filename}")
    print("=" * 70)

    file_path = os.path.abspath(filename)

    extension = os.path.splitext(filename)[1].lower()


    # ========================================================
    # CASE 1 — PDF
    # ========================================================

    if extension == ".pdf":

        print("PDF detected.")
        print("Converting PDF pages to images...")

        invoice_images = pdf_to_images(
            file_path,
            dpi=200
        )

        print(
            f"PDF converted successfully: "
            f"{len(invoice_images)} page(s)"
        )


    # ========================================================
    # CASE 2 — IMAGE
    # ========================================================

    elif extension in [
        ".jpg",
        ".jpeg",
        ".png",
        ".webp",
        ".bmp",
        ".tiff",
        ".tif"
    ]:

        print("Image detected.")

        invoice_images = [
            Image.open(file_path).convert("RGB")
        ]


    # ========================================================
    # UNSUPPORTED FILE
    # ========================================================

    else:

        raise ValueError(
            f"Unsupported file type: {extension}\n"
            "Please upload a PDF or image."
        )


    # ========================================================
    # DISPLAY FIRST PAGE
    # ========================================================

    print("\nInvoice preview:")

    display(invoice_images[0])


    # ========================================================
    # RUN INFERENCE
    # ========================================================

    print("\n" + "=" * 70)
    print("RUNNING TWO-STAGE PIPELINE")
    print("=" * 70)

    print("""
Stage 1 → Qwen3-VL-4B Invoice Extraction
Stage 2 → Qwen3-4B-Instruct Accounting Categorization
Stage 3 → Merge
""")


    # ========================================================
    # SINGLE PAGE
    # ========================================================

    if len(invoice_images) == 1:

        # Save converted image temporarily
        temp_image_path = "/content/invoice_page_1.png"

        invoice_images[0].save(
            temp_image_path,
            format="PNG"
        )

        final_result = run_invoice_inference(
            temp_image_path
        )


    # ========================================================
    # MULTI-PAGE PDF
    # ========================================================

    else:

        print(
            f"Multi-page invoice detected: "
            f"{len(invoice_images)} pages"
        )

        page_results = []

        for page_number, image in enumerate(
            invoice_images,
            start=1
        ):

            print("\n" + "-" * 60)
            print(f"Processing page {page_number}")
            print("-" * 60)

            temp_image_path = (
                f"/content/invoice_page_{page_number}.png"
            )

            image.save(
                temp_image_path,
                format="PNG"
            )

            result = run_invoice_inference(
                temp_image_path
            )

            page_results.append(result)


        # For multi-page PDFs, return all page results
        final_result = {
            "pages": page_results
        }


    # ========================================================
    # FINAL JSON
    # ========================================================

    print("\n" + "=" * 70)
    print("FINAL JSON")
    print("=" * 70)

    display(
        JSON(final_result)
    )

## 11. Test one invoice

Upload a PDF or image; PDFs get auto-converted to PNG at 150 DPI (avoids the
earlier OOM we hit from high-DPI renders).

In [ ]:
from google.colab import userdata

NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

# Example single-invoice test (Stage 1 -> Stage 2 -> Stage 3)
# image_path = "/content/your_invoice.png"
# result = run_two_stage_invoice(image_path)
# print(json.dumps(result, indent=2, ensure_ascii=False))

!pip install pyngrok
# ============================================================
# FastAPI + ngrok for Backend Integration
# ============================================================

import os
import io
import json
import base64
import tempfile
import traceback

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from PIL import Image

import nest_asyncio
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()

app = FastAPI(
    title="Qwen3 Invoice API"
)

# -------------------------------
# Request Model
# -------------------------------

from typing import List, Dict, Any, Optional

class InvoiceRequest(BaseModel):
    image_base64: str
    # Optional live Zoho COA/tax lists allow a single extraction request
    # to perform Stage-2 classification as well. Existing callers can omit them.
    chart_of_accounts: List[Dict[str, Any]] = []
    available_taxes: List[Dict[str, Any]] = []


class CategorizeRequest(BaseModel):
    line_items: List[Dict[str, Any]]
    chart_of_accounts: List[Dict[str, Any]]
    vendor_name: Optional[str] = None
    # Optional: send Zoho's live tax records so Qwen3-4B can map the
    # extracted GST details to the exact Zoho tax record without inventing one.
    available_taxes: List[Dict[str, Any]] = []
    # Optional invoice-level totals/context. The endpoint can work without it
    # because the line items already carry their own CGST/SGST/IGST details.
    invoice_context: Optional[Dict[str, Any]] = None


# -------------------------------
# Health
# -------------------------------

@app.get("/")
async def root():
    return {"message": "Qwen3 Invoice API Running"}


@app.get("/health")
async def health():
    return {"status": "ok"}


# -------------------------------
# Accounting Categorization
# -------------------------------

@app.post("/api/infer/categorize-accounting")
async def categorize_accounting_endpoint(req: CategorizeRequest):
    try:
        print("=" * 60)
        print("Received accounting categorization request")
        results = categorize_line_items(
            line_items=req.line_items,
            chart_of_accounts=req.chart_of_accounts,
            vendor_name=req.vendor_name,
            invoice_context=req.invoice_context or {},
            available_taxes=req.available_taxes,
        )
        print("Categorization complete")
        return results
    except Exception as e:
        traceback.print_exc()
        raise HTTPException(
            status_code=500,
            detail=str(e)
        )


# -------------------------------
# Invoice Extraction
# -------------------------------

@app.post("/api/infer/extract-invoice")
async def process_invoice(req: InvoiceRequest):

    temp_path = None
    pdf_path = None

    try:

        print("=" * 60)
        print("Received inference request")

        # Decode Base64
        image_bytes = base64.b64decode(req.image_base64)

        print("Decoded bytes:", len(image_bytes))

        # -------------------------------------------------------
        # PDF
        # -------------------------------------------------------
        if image_bytes[:4] == b"%PDF":

            print("Detected PDF")

            import fitz

            with tempfile.NamedTemporaryFile(
                delete=False,
                suffix=".pdf"
            ) as tmp_pdf:

                tmp_pdf.write(image_bytes)
                pdf_path = tmp_pdf.name

            doc = fitz.open(pdf_path)

            page = doc.load_page(0)

            pix = page.get_pixmap(dpi=200)

            with tempfile.NamedTemporaryFile(
                delete=False,
                suffix=".png"
            ) as tmp_img:

                temp_path = tmp_img.name

            pix.save(temp_path)

            doc.close()

            print("Converted PDF -> PNG:", temp_path)

        # -------------------------------------------------------
        # Image (.png/.jpg/.jpeg)
        # -------------------------------------------------------
        else:

            print("Detected Image")

            image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

            ext = ".png"

            if image.format == "JPEG":
                ext = ".jpg"

            with tempfile.NamedTemporaryFile(
                delete=False,
                suffix=ext
            ) as tmp_img:

                temp_path = tmp_img.name

            image.save(temp_path)

            print("Saved image:", temp_path)

        print("Running two-stage agentic invoice pipeline")

        result = run_two_stage_invoice(
            temp_path,
            chart_of_accounts=req.chart_of_accounts,
            available_taxes=req.available_taxes,
        )

        print("Inference complete")

        # Cleanup
        if temp_path and os.path.exists(temp_path):
            os.remove(temp_path)

        if pdf_path and os.path.exists(pdf_path):
            os.remove(pdf_path)

        return result

    except Exception as e:

        traceback.print_exc()

        if temp_path and os.path.exists(temp_path):
            os.remove(temp_path)

        if pdf_path and os.path.exists(pdf_path):
            os.remove(pdf_path)

        raise HTTPException(
            status_code=500,
            detail=str(e)
        )

# ============================================================
# Start ngrok
# ============================================================

try:
    ngrok.kill()
except:
    pass

# Use the NGROK_AUTH_TOKEN variable
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(8000).public_url

print("=" * 70)
print("Public URL   :", public_url)
print("Health API   :", public_url + "/health")
print("Invoice API  :", public_url + "/api/infer/extract-invoice")
print("Categorize   :", public_url + "/api/infer/categorize-accounting")
print("=" * 70)

# ============================================================
# Start FastAPI
# ============================================================

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(config)

import asyncio
asyncio.get_event_loop().run_until_complete(server.serve())